# Partie 2 **Fracture Classification Dataset**

Ce Dataset diponible sur kaggle comprend **4083** images dont **3366** images de classe 0 (Sain) et **717** images de classe 1 (Fracture).

Par conséquent, il est important de noter qu'un algorithme se contenant de répondre simplement comment étant sain pour chaque image aura une accuracy de **82,4%**. Cela implique plusieurs changements dans l'entrainement de nos modèles dont la métrique de validation car l'accuracy seul ne sera plus pertinente et l'erreur attribuée aux images comprenant une fracture et etant mal classé devra être plus grande.

## Analyse exploratoire du dataset

### Analyse de la distribution du dataset

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = pd.read_csv('dataset.csv')

In [ ]:
sns.countplot(df, x='fractured' )
plt.show()

In [ ]:
colonnes = ['leg', 'hand', 'hip', 'shoulder', 'mixed', 'hardware', 'multiscan']
df.groupby('fractured')[colonnes].sum().T.plot(kind='bar', figsize=(10, 6), color=['skyblue', 'salmon'])
plt.show()

Lorsqu'une radio est étiquetée avec **hardware = 1**, cela signifie que l'on voit clairement sur l'image :

* Des vis et des plaques d'ostéosynthèse (utilisées pour ressouder les morceaux d'un os cassé).

* Des broches ou des tiges métalliques intra-médullaires (glissées au centre des os longs comme le fémur ou le tibia).

* Des prothèses complètes ou partielles (comme une prothèse de hanche ou d'épaule).

**mixed** correspond au radio qui englobe plusieurs parties du corps. **multiscan** indique si plusieurs radios sur la même photo.

In [ ]:
print(df.isna().sum())
print(f"Il y a {df['multiscan'].sum()} d'images contennants plusieurs radios")

In [ ]:
import matplotlib.pyplot as plt
import cv2
import os

df_exemples = df[(df['fractured'] == 1) & (df['multiscan']==1)].sample(4)
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle("Exemples de Radiographies avec Fractures Visibles", fontsize=16, fontweight='bold', y=1.05)
for ax, (index, row) in zip(axes, df_exemples.iterrows()):
    nom_image = row['image_id']
    chemin_complet = os.path.join("images\Fractured", nom_image)
    img = cv2.imread(chemin_complet, cv2.IMREAD_GRAYSCALE)
    ax.imshow(img, cmap='gray')
    ax.axis('off')
plt.show()


On voit bien ici le problème des multiscans...

On s'occupe ici de charger les images, des les redimensionner en 224*224 et on les encode sur un seul canal (niveau de gris).

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

# 1. Renseigne les chemins vers tes deux dossiers
dossier_sain = "images/Non_fractured"
dossier_fractured = "images/Fractured"
compteur= 0
# Configuration des dossiers avec leurs couleurs respectives pour le graphique
dossiers = {
    "Sain": (dossier_sain, 'mediumseagreen'),
    "Fracturé": (dossier_fractured, 'tomato')
}

plt.figure(figsize=(12, 7))

# 2. Boucle pour analyser chaque dossier l'un après l'autre
for nom_classe, (chemin_dossier, couleur) in dossiers.items():
    largeurs = []
    hauteurs = []

    print(f"Analyse du dossier : {nom_classe}...")
    
    # Vérification basique pour éviter les erreurs de chemin
    if not os.path.exists(chemin_dossier):
        print(f" /!\\ Attention, le dossier est introuvable : {chemin_dossier}")
        continue

    for nom_fichier in os.listdir(chemin_dossier):
        if nom_fichier.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.bmp')):
            chemin_complet = os.path.join(chemin_dossier, nom_fichier)
            try:
                with Image.open(chemin_complet) as img:
                    largeur, hauteur = img.size
                    largeurs.append(largeur)
                    hauteurs.append(hauteur)
                    compteur+=1
            except Exception as e:
                print(f"Impossible de lire l'image {nom_fichier}: {e}")

    # 3. Ajout des points sur le graphique pour la classe en cours
    # On ajoute len(largeurs) dans le label pour afficher le nombre d'images de chaque classe
    plt.scatter(
        largeurs, hauteurs, 
        alpha=0.5, color=couleur, edgecolors='white', linewidth=0.5,
        label=f'{nom_classe} ({len(largeurs)} images)'
    )

# 4. Mise en forme finale du graphique
plt.title("Comparaison des résolutions d'images (Sain vs Fracturé)", fontsize=14, fontweight='bold')
plt.xlabel("Largeur (pixels)")
plt.ylabel("Hauteur (pixels)")
plt.grid(True, linestyle='--', alpha=0.5)

# Seuil de basse qualité
seuil = 150
plt.axvline(x=seuil, color='black', linestyle=':', label=f'Seuil bas ({seuil}px)')
plt.axhline(y=seuil, color='black', linestyle=':')

plt.legend()
plt.tight_layout()
plt.show()
print(compteur)

In [ ]:
import os
import pandas as pd
from PIL import Image

# 1. Chemins vers tes dossiers
dossier_sain = "images/Non_fractured"
dossier_fractured = "images/Fractured"

dossiers = {
    "Sain": dossier_sain,
    "Fracturé": dossier_fractured
}

# Seuil de recherche
seuil_largeur = 220
seuil_hauteur = 220

# Compteurs pour le résultat final
petites_images_par_classe = {"Sain": 0, "Fracturé": 0}

# Liste pour stocker les informations et créer le nouveau CSV complet
donnees_completes = []

print("Lecture des en-têtes d'images en cours...")

for nom_classe, chemin_dossier in dossiers.items():
    if not os.path.exists(chemin_dossier):
        print(f" /!\\ Dossier introuvable : {chemin_dossier}")
        continue

    for nom_fichier in os.listdir(chemin_dossier):
        # Filtrer uniquement les formats d'images valides
        if nom_fichier.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.bmp')):
            chemin_complet = os.path.join(chemin_dossier, nom_fichier)
            
            try:
                # Open n'ouvre que l'en-tête (très rapide et léger en RAM)
                with Image.open(chemin_complet) as img:
                    largeur, hauteur = img.size
                    
                    # Sauvegarde des infos pour le futur CSV
                    donnees_completes.append({
                        "Fichier": nom_fichier,
                        "Classe": nom_classe,
                        "Largeur": largeur,
                        "Hauteur": hauteur
                    })
                    
                    # Vérification du critère 
                    if largeur < seuil_largeur or hauteur < seuil_hauteur:
                        petites_images_par_classe[nom_classe] += 1
                        
            except Exception as e:
                # Permet d'ignorer et lister un éventuel fichier corrompu
                print(f"Erreur de lecture sur le fichier {nom_fichier} : {e}")

# 2. Affichage du bilan textuel personnalisé
print(f"\n=== BILAN DES IMAGES INFÉRIEURES À {seuil_largeur}x{seuil_hauteur} ===")
total_petites = 0
for classe, compteur in petites_images_par_classe.items():
    print(f"-> Classe [{classe}] : {compteur} images concernées.")
    total_petites += compteur

print(f"Total général d'images trop petites : {total_petites}")

# 3. Sauvegarde automatique dans un nouveau CSV enrichi
df_nouveau = pd.DataFrame(donnees_completes)
chemin_csv_sortie = "dataset_images_dimensions.csv"
df_nouveau.to_csv(chemin_csv_sortie, index=False)

print(f"\n[Succès] Un nouveau fichier CSV contenant toutes les dimensions a été créé : '{chemin_csv_sortie}'")

On voit ici que les images sont en moyenne d'assez bonne résolution pour pouvoir être étudiées.

## Nettoyage du dataset 

On s'est d'abord proposé de découper en deux toutes les images des multiscans. Ce que nous n'avions pas remarqué au début, c'est que pourles images fracturées, il est possible qu'une des deux images soients présente bien une vue avec une fracture mais que la deuxième non. Ce problème fait que en séparant les deux images et en les mettant dans la classe Fractured,une des deux a potentiellement été mal classée.

On décide donc au final de ne pas séparer les multiscans et de les laisser dans leur forme originale.

In [ ]:
import os
import cv2
import pandas as pd
dossier_principal = "images" 

def rendre_carre_opencv(img):
    hauteur, largeur = img.shape
    max_cote = max(hauteur, largeur)
    
    # Calcul de l'épaisseur des bordures
    haut = (max_cote - hauteur) // 2
    bas = max_cote - hauteur - haut
    gauche = (max_cote - largeur) // 2
    droite = max_cote - largeur - gauche
    
    # Ajout des bordures noires (valeur 0)
    img_carree = cv2.copyMakeBorder(img, haut, bas, gauche, droite, cv2.BORDER_CONSTANT, value=0)
    
    return img_carree


df_multiscan = df[df['multiscan'] == 1]
cpt = 0


for index, row in df_multiscan.iterrows():
    nom_fichier = row['image_id']
    statut_medical = row['fractured']

    if statut_medical == 1:
        nom_sous_dossier = "Fractured"
        chemin_complet = os.path.join(dossier_principal, 'Fractured', nom_fichier)
    else:
        nom_sous_dossier = "Non_fractured"
        chemin_complet = os.path.join(dossier_principal, 'Non_fractured', nom_fichier)
    
    img = cv2.imread(chemin_complet, cv2.IMREAD_GRAYSCALE)
    
    if img is None: print("Manquant:", chemin_complet); continue
       
    hauteur, largeur = img.shape
    milieu_x = largeur // 2
    
    img_gauche = img[:, :milieu_x]
    img_droite = img[:, milieu_x:]
    
    #Création des nouveaux noms de fichiers
    img_gauche = rendre_carre_opencv(img_gauche)
    img_droite = rendre_carre_opencv(img_droite)

    

    nom_base, extension = os.path.splitext(nom_fichier)
    nom_gauche = f"{nom_base}_gauche{extension}"
    nom_droite = f"{nom_base}_droite{extension}"
    
    chemin_gauche = os.path.join(dossier_principal,nom_sous_dossier, nom_gauche)
    chemin_droite = os.path.join(dossier_principal,nom_sous_dossier, nom_droite)
    
    # 5. Sauvegarde des nouvelles images sur le disque dur
   
    cv2.imwrite(chemin_gauche, img_gauche)
    cv2.imwrite(chemin_droite, img_droite)
    
    cpt += 1

print(f" {cpt} images multiscan ont été coupées en deux, créant {cpt * 2} nouvelles images individuelles.")

On vérifie ce qu'a donné le decoupage avec l'ajout de bande noir sur les côtés pour que les images soient bien carrées.

In [ ]:
import matplotlib.pyplot as plt
import cv2
import os
import random

images_decoupees = [f for f in os.listdir("images\Fractured") if f.endswith('_gauche.jpg') or f.endswith('_droite.jpg')]
print(len(images_decoupees))
exemples = random.sample(images_decoupees, 4)

# 4. Affichage
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle("Vérification des Multiscans Découpés et Cadrés (Fractures)", fontsize=16, fontweight='bold', y=1.05)

for ax, nom_image in zip(axes, exemples):
    chemin_complet = os.path.join('images\Fractured', nom_image)
    
    img = cv2.imread(chemin_complet, cv2.IMREAD_GRAYSCALE)
    
    ax.imshow(img, cmap='gray')
    # On ajoute le nom de l'image pour vérifier que c'est bien une _gauche ou _droite
    ax.set_title(nom_image, fontsize=9) 
    ax.axis('off')

plt.tight_layout()
plt.show()

On va maintenant supprimer les images multiscan

In [ ]:
import os
import shutil # Bibliothèque Python pour copier/déplacer des fichiers
import pandas as pd

dossier_corbeille = "multiscan_archives"
os.makedirs(dossier_corbeille, exist_ok=True)

df_multiscan = df[df['multiscan'] == 1]
cpt = 0

for index, row in df_multiscan.iterrows():
    nom_fichier = row['image_id']
    statut_medical = row['fractured']
    if statut_medical == 1:
        chemin_original = os.path.join(dossier_principal, 'Fractured', nom_fichier)
    else:
        chemin_original = os.path.join(dossier_principal, 'Non_fractured', nom_fichier)
    
    if os.path.exists(chemin_original):
        chemin_archive = os.path.join(dossier_corbeille, nom_fichier)
        
        # On déplace l'image sans la supprimer
        shutil.move(chemin_original, chemin_archive)        
        cpt += 1
    else:
        pass # L'image a déjà été déplacée ou supprimée lors d'un test précédent

print(f"{cpt} images originales ont été retirées de l'entraînement.")

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np


dossier_principal = "images" 
transformations = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

dataset_complet = datasets.ImageFolder(root=dossier_principal, transform=transformations)

print(f"Mapping des classes : {dataset_complet.class_to_idx}")
print(f"Nombre total d'images chargées : {len(dataset_complet)}")

dataloader = DataLoader(dataset_complet, batch_size=16, shuffle=True)



## Entrainement sans augentation de données

#### Création des dataloader train et test

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from torch.utils.data import WeightedRandomSampler
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix, classification_report
import random

seed = 123
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

dossier_principal = "images" 
if torch.xpu.is_available():
    device = torch.device("xpu")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(seed)
else:
    device = torch.device("cpu")
print(device)

In [ ]:
batch_size = 64  
img_size = (224, 224)
val_split = 0.2
seed = 123

dossier_principal = "images" 
transformations = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset_complet = datasets.ImageFolder(root=dossier_principal, transform=transformations)



print(f"Mapping des classes : {dataset_complet.class_to_idx}")
print(f"Nombre total d'images chargées : {len(dataset_complet)}")


indices = list(range(len(dataset_complet)))
labels = dataset_complet.targets


idx_train_val, idx_test = train_test_split(
    indices, 
    test_size=0.15, 
    random_state=seed, 
    stratify=labels
)

# On coupe les 85% restants en Train (70%) et Val (15%)
# (Pour isoler 15% du total à partir des 85% restants, le ratio est 15/85 = 0.176)
labels_train_val = [labels[i] for i in idx_train_val]

idx_train, idx_val = train_test_split(
    idx_train_val, 
    test_size=0.176, 
    random_state=seed, 
    stratify=labels_train_val
)

# 4. Création des 3 Sous-Datasets
dataset_train = Subset(datasets.ImageFolder(root=dossier_principal, transform=transformations), idx_train)
dataset_val = Subset(datasets.ImageFolder(root=dossier_principal, transform=transformations), idx_val)
dataset_test = Subset(datasets.ImageFolder(root=dossier_principal, transform=transformations), idx_test)



# Création des DataLoaders finaux
train_loader = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset_val,batch_size=batch_size, shuffle=False)
test_loader = DataLoader(dataset_test, batch_size=batch_size, shuffle=False)

### RESNET-50

#### Optuna sur ResNet50

In [ ]:
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from sklearn.metrics import f1_score
from PIL import ImageFile

# Cette ligne demande à PIL d'être tolérant avec les images corrompues
ImageFile.LOAD_TRUNCATED_IMAGES = True



EPOQUES_PAR_TRIAL = 4 # On garde ça court pour l'exploration

labels_train = [dataset_complet.targets[i] for i in idx_train]

# 2. Scikit-Learn calcule les poids "balanced" (inversément proportionnels)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels_train),
    y=labels_train
)

# On convertit ce tableau NumPy en Tenseur PyTorch et on l'envoie sur le GPU
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)


def objective(trial):
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.7)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)
    freeze_layer = trial.suggest_categorical("freeze_layer", ["layer4.2", "layer4.1", "layer4.0"])

    modele = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    
    unfreeze_from = {
        "layer4.2": ["layer4.2.", "fc."],
        "layer4.1": ["layer4.1.", "layer4.2.", "fc."],
        "layer4.0": ["layer4.0.", "layer4.1.", "layer4.2.", "fc."]
    }

    prefixes = unfreeze_from[freeze_layer]

    for name, param in modele.named_parameters():
        if any(name.startswith(p) for p in prefixes):
            param.requires_grad = True
        else:
            param.requires_grad = False 

    nb_features_entree = modele.fc.in_features
    modele.fc = nn.Sequential(
        nn.Dropout(p=dropout_rate), 
        nn.Linear(nb_features_entree, 2)
    )
    modele = modele.to(device)

    # Injection du LR et du Weight Decay trouvés par Optuna
    parametres_a_entrainer = filter(lambda p: p.requires_grad, modele.parameters())
    optimiseur = optim.Adam(parametres_a_entrainer, lr=lr, weight_decay=weight_decay)
    
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

    for epoque in range(EPOQUES_PAR_TRIAL):
        
        modele.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimiseur.zero_grad()
            predictions = modele(images)
            perte = criterion(predictions, labels)
            perte.backward()
            optimiseur.step()
            
        # Validation
        modele.eval()
        toutes_les_predictions = []
        tous_les_vrais_labels = []
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                predictions = modele(images)
                _, classes_predites = torch.max(predictions, 1)
                
                toutes_les_predictions.extend(classes_predites.cpu().numpy())
                tous_les_vrais_labels.extend(labels.cpu().numpy())
                
        # Calcul du F1-Score à cette époque
        f1 = f1_score(tous_les_vrais_labels, toutes_les_predictions, pos_label=1, zero_division=0)
        

        # On signale le F1-Score intermédiaire à Optuna
        trial.report(f1, epoque)
        
        # Si Optuna détecte que cet essai est catastrophique comparé aux précédents, il coupe !
        if trial.should_prune():
            raise optuna.TrialPruned()

    # Optuna a besoin qu'on lui renvoie la valeur à optimiser 
    return f1


# On veut maximiser le F1-Score
study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner(n_warmup_steps=1))

study.optimize(objective, n_trials=20)

print(f"Meilleur F1-Score obtenu : {study.best_value*100:.2f}%")
print("Meilleurs hyperparamètres :", study.best_params)


##### Visualisation des paramètres trouvés par Optuna

In [ ]:
from optuna.visualization import plot_optimization_history
from optuna.visualization import plot_param_importances
from optuna.visualization import plot_slice
import matplotlib.pyplot as plt

fig_history = plot_optimization_history(study)
fig_history.show()

fig_importances = plot_param_importances(study)
fig_importances.show()

fig_slice = plot_slice(study)
fig_slice.show()

#### Entrainement de ResNet-50 avec les meilleurs hyperamètres trouvées par Optuna

In [ ]:
meilleurs_params = study.best_params
meilleur_lr = meilleurs_params["lr"]
meilleur_dropout = meilleurs_params["dropout_rate"]
meilleur_wd = meilleurs_params["weight_decay"]
meilleur_degel = meilleurs_params["freeze_layer"]

epoques_finales = 20
patience = 3
patience_counter = 0
meilleur_f1 = 0.0

modele_RN50_final = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

unfreeze_from = {
    "layer4.2": ["layer4.2.", "fc."],
    "layer4.1": ["layer4.1.", "layer4.2.", "fc."],
    "layer4.0": ["layer4.0.", "layer4.1.", "layer4.2.", "fc."]
}
prefixes = unfreeze_from[meilleur_degel]
for name, param in modele_RN50_final.named_parameters():
    if any(name.startswith(p) for p in prefixes):
        param.requires_grad = True
    else:
        param.requires_grad = False

nb_features_entree = modele_RN50_final.fc.in_features
modele_RN50_final.fc = nn.Sequential(
    nn.Dropout(p=meilleur_dropout), 
    nn.Linear(nb_features_entree, 2)
)
modele_RN50_final = modele_RN50_final.to(device)


labels_train = [dataset_train.dataset.targets[i] for i in dataset_train.indices]
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels_train),
    y=labels_train
)

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
parametres_a_entrainer = filter(lambda p: p.requires_grad, modele_RN50_final.parameters())
optimiseur = optim.Adam(parametres_a_entrainer, lr=meilleur_lr, weight_decay=meilleur_wd)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
meilleur_f1 = 0.0

print("Lancement de l'entraînement")

for epoque in range(epoques_finales):
    
    modele_RN50_final.train()
    perte_train_totale = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimiseur.zero_grad()
        predictions = modele_RN50_final(images)
        perte = criterion(predictions, labels)
        perte.backward()
        optimiseur.step()
        
        perte_train_totale += perte.item() 
    moyenne_perte_train = perte_train_totale / len(train_loader)
    
    #Validation
    modele_RN50_final.eval()
    perte_val_totale = 0.0
    toutes_les_predictions = []
    tous_les_vrais_labels = []
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            
            output  = modele_RN50_final(images)
            perte = criterion(output, labels)
            perte_val_totale += perte.item()
            
            _, classes_predites = torch.max(output, 1)
            
            toutes_les_predictions.extend(classes_predites.cpu().numpy())
            tous_les_vrais_labels.extend(labels.cpu().numpy())
            
    moyenne_perte_val = perte_val_totale / len(val_loader)
    
    # Calcule des scores
    accuracy = accuracy_score(tous_les_vrais_labels, toutes_les_predictions)
    rappel = recall_score(tous_les_vrais_labels, toutes_les_predictions, pos_label=1, zero_division=0)
    f1 = f1_score(tous_les_vrais_labels, toutes_les_predictions, pos_label=1, zero_division=0)
    
    print(f"Époque [{epoque+1}/{epoques_finales}]")
    print(f"   Train Loss: {moyenne_perte_train:.4f} | Val Loss: {moyenne_perte_val:.4f}")
    print(f"   Accuracy : {accuracy*100:.2f}%")
    print(f"   Rappel (Sensibilité Fractures) : {rappel*100:.2f}%")
    print(f"   F1-Score : {f1*100:.2f}%")
    
    # Sauvegarde du meilleur modèle basé sur le F1-Score (compromis idéal)
    if f1 > meilleur_f1:
        meilleur_f1 = f1
        torch.save(modele_RN50_final.state_dict(), 'meilleur_modele_ResNet50.pth')
        print("Nouveau meilleur modèle baseline sauvegardé !")
        patience_counter = 0
    else: 
        patience_counter+=1
    
    if patience_counter >= patience:
        print(f"\nEarly Stopping déclenché à l'époque {epoque+1}.")
        break

#### Test du meilleur modèle de ResNet50

In [ ]:
modele_RN50_test = models.resnet50(weights=None)

modele_RN50_test.load_state_dict(torch.load('meilleur_modele_ResNet50.pth', map_location=device))

nb_features_entree = modele_RN50_test.fc.in_features
modele_RN50_test.fc = nn.Sequential(
    nn.Dropout(p=meilleur_dropout), 
    nn.Linear(nb_features_entree, 2)
)
modele_RN50_test = modele_RN50_test.to(device)
modele_RN50_test.eval()





predictions = []
vrais_labels = []

# torch.no_grad() coupe le calcul des gradients (économise la RAM et accélère le processus)
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        
        pred = modele_RN50_test(images)
        _, classes_predites = torch.max(pred, 1)
        
        predictions.extend(classes_predites.cpu().numpy())
        vrais_labels.extend(labels.cpu().numpy())


acc_test = accuracy_score(vrais_labels, predictions)
rappel_test = recall_score(vrais_labels, predictions, pos_label=1, zero_division=0)
f1_test = f1_score(vrais_labels, predictions, pos_label=1, zero_division=0)


print(f"Accuracy Globale  : {acc_test*100:.2f}%")
print(f"Rappel (Sensibilité): {rappel_test*100:.2f}% ")
print(f"F1-Score          : {f1_test*100:.2f}%")


# Affichage du rapport détaillé de Scikit-Learn
noms_classes = ['Sain (0)', 'Fracture (1)']
print("\nRAPPORT DE CLASSIFICATION :")
print(classification_report(vrais_labels, predictions, target_names=noms_classes))


cm = confusion_matrix(vrais_labels, predictions)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=noms_classes, 
            yticklabels=noms_classes,
            annot_kws={"size": 14})

plt.title('Matrice de Confusion (Jeu de Test)', fontsize=16, pad=20)
plt.ylabel('Vérité', fontsize=12)
plt.xlabel('Prédiction', fontsize=12)
plt.show()

if device.type == "xpu":
    torch.xpu.empty_cache()
elif device.type == "cuda":
    torch.cuda.empty_cache()

## Entrainement avec augmentation de données

In [ ]:
# Configuration de base

def preparer_dataloaders(img_size=(224, 224), batch_size=32, seed=123, dossier_principal="images"):

    transformations_train = transforms.Compose([
        transforms.Grayscale(num_output_channels=3),
        transforms.Resize(img_size),
        

        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
        
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    transformations_val = transforms.Compose([
        transforms.Grayscale(num_output_channels=3),
        transforms.Resize(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])




    # Chargement initial pour obtenir la structure
    dataset_complet = datasets.ImageFolder(root=dossier_principal)
    indices = list(range(len(dataset_complet)))
    labels_complets = dataset_complet.targets

    # Découpe stratifiée
    idx_train_val, idx_test = train_test_split(
        indices, test_size=0.15, random_state=seed, stratify=labels_complets
    )

    # On coupe les 85% restants en Train (70%) et Val (15%)
    # Pour isoler 15% du total à partir des 85% restants, le ratio est 15/85 = 0.176)
    labels_train_val = [labels_complets[i] for i in idx_train_val]

    idx_train, idx_val = train_test_split(
        idx_train_val, test_size=0.176, random_state=seed, stratify=labels_train_val
    )

    # Création des sous-datasets avec leurs transformations respectives
    dataset_train = Subset(datasets.ImageFolder(root=dossier_principal, transform=transformations_train), idx_train)
    dataset_val = Subset(datasets.ImageFolder(root=dossier_principal, transform=transformations_val), idx_val)
    dataset_test = Subset(datasets.ImageFolder(root=dossier_principal, transform=transformations_val), idx_test)

    # corection de l'équilibre des classes
    # Correction de la fuite de données : extraction stricte des labels du subset train
    labels_train = [dataset_complet.targets[i] for i in idx_train]
    compte_classes = np.bincount(labels_train) # [nb_sains, nb_fractures]

    # Poids inversement proportionnel pour chaque image
    poids_par_classe = 1.0 / compte_classes
    poids_images = [poids_par_classe[label] for label in labels_train]

    # Le sampler va forcer une distribution 50% / 50% dans chaque batch
    sampler_equilibre = WeightedRandomSampler(
        weights=poids_images, 
        num_samples=len(poids_images), 
        replacement=True
    )

    # Remplacement du sampler dans le DataLoader d'entraînement (shuffle doit être absent si sampler est défini)
    train_loader = DataLoader(dataset_train, batch_size=batch_size, sampler=sampler_equilibre)
    val_loader = DataLoader(dataset_val, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(dataset_test,batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

### Optuna pour les 3 architectures retenues : Resnet50, V2s, denseNet

On a décider d'étudier à nouveau Resnet50 pour pouvoir comparer l'effet de l'augmentation de données. Puis nous avons choisi Efficientnet V2s et densenet. Densenet est un modèle lourd ou chaque couche est reliée à toutes les autres par concaténation. Il a justement été utilisé dans le domaine médicale.

#### Optuna Resnet-50

In [ ]:
EPOQUES_PAR_TRIAL = 4
train_loader, val_loader, test_loader = preparer_dataloaders(img_size=(224, 224), batch_size=32)
def objective_resnet(trial):
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.7)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)
    freeze_layer = trial.suggest_categorical("freeze_layer", ["layer4.2", "layer4.1", "layer4.0"])

    modele = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    
    unfreeze_from = {
        "layer4.2": ["layer4.2.", "fc."],
        "layer4.1": ["layer4.1.", "layer4.2.", "fc."],
        "layer4.0": ["layer4.0.", "layer4.1.", "layer4.2.", "fc."]
    }
    prefixes = unfreeze_from[freeze_layer]

    for name, param in modele.named_parameters():
        param.requires_grad = any(name.startswith(p) for p in prefixes)

    nb_features = modele.fc.in_features
    modele.fc = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(nb_features, 2))
    modele = modele.to(device)

    optimiseur = optim.Adam(filter(lambda p: p.requires_grad, modele.parameters()), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    for epoque in range(EPOQUES_PAR_TRIAL):
        modele.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimiseur.zero_grad()
            perte = criterion(modele(images), labels)
            perte.backward()
            optimiseur.step()
            
        modele.eval()
        toutes_preds, tous_labels = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                _, preds = torch.max(modele(images), 1)
                toutes_preds.extend(preds.cpu().numpy())
                tous_labels.extend(labels.cpu().numpy())
                
        f1 = f1_score(tous_labels, toutes_preds, pos_label=1, zero_division=0)
        trial.report(f1, epoque)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return f1

study_resnet = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner(n_warmup_steps=1))
study_resnet.optimize(objective_resnet, n_trials=15)
print(f"Meilleur F1 ResNet-50 : {study_resnet.best_value*100:.2f}% | Params: {study_resnet.best_params}")

#### Entrainement de ResNet50 avec les meilleurs hyeramètres trouvés

In [ ]:
meilleurs_params = study.best_params
meilleur_lr = meilleurs_params["lr"]
meilleur_dropout = meilleurs_params["dropout_rate"]
meilleur_wd = meilleurs_params["weight_decay"]
meilleur_degel = meilleurs_params["freeze_layer"]

epoques_finales = 20
patience = 3
patience_counter = 0
meilleur_f1 = 0.0

modele_RN50_final = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

unfreeze_from = {
    "layer4.2": ["layer4.2.", "fc."],
    "layer4.1": ["layer4.1.", "layer4.2.", "fc."],
    "layer4.0": ["layer4.0.", "layer4.1.", "layer4.2.", "fc."]
}
prefixes = unfreeze_from[meilleur_degel]
for name, param in modele_RN50_final.named_parameters():
    if any(name.startswith(p) for p in prefixes):
        param.requires_grad = True
    else:
        param.requires_grad = False

nb_features_entree = modele_RN50_final.fc.in_features
modele_RN50_final.fc = nn.Sequential(
    nn.Dropout(p=meilleur_dropout), 
    nn.Linear(nb_features_entree, 2)
)
modele_RN50_final = modele_RN50_final.to(device)
criterion = nn.CrossEntropyLoss()
parametres_a_entrainer = filter(lambda p: p.requires_grad, modele_RN50_final.parameters())
optimiseur = optim.Adam(parametres_a_entrainer, lr=meilleur_lr, weight_decay=meilleur_wd)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimiseur, mode='max', patience=2, factor=0.5, verbose=True)
meilleur_f1 = 0.0

print("Lancement de l'entraînement")

for epoque in range(epoques_finales):
    
    modele_RN50_final.train()
    perte_train_totale = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimiseur.zero_grad()
        output = modele_RN50_final(images)
        perte = criterion(output, labels)
        perte.backward()
        optimiseur.step()
        
        perte_train_totale += perte.item() 
    moyenne_perte_train = perte_train_totale / len(train_loader)
    
    #Validation
    modele_RN50_final.eval()
    perte_val_totale = 0.0
    predictions = []
    vrais_labels = []
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            
            output = modele_RN50_final(images)
            perte = criterion(output, labels)
            perte_val_totale += perte.item()
            
            _, classes_predites = torch.max(output, 1)
            
            predictions.extend(classes_predites.cpu().numpy())
            vrais_labels.extend(labels.cpu().numpy())
            
    moyenne_perte_val = perte_val_totale / len(val_loader)
    
    # Calcule des scores
    accuracy = accuracy_score(vrais_labels, predictions)
    rappel = recall_score(vrais_labels, predictions, pos_label=1, zero_division=0)
    f1 = f1_score(vrais_labels, predictions, pos_label=1, zero_division=0)
    
    print(f"Époque [{epoque+1}/{epoques_finales}]")
    print(f"   Train Loss: {moyenne_perte_train:.4f} | Val Loss: {moyenne_perte_val:.4f}")
    print(f"   Accuracy : {accuracy*100:.2f}%")
    print(f"   Rappel (Sensibilité Fractures) : {rappel*100:.2f}%")
    print(f"   F1-Score : {f1*100:.2f}%")

    scheduler.step(f1)
    
    # Sauvegarde du meilleur modèle basé sur le F1-Score (compromis idéal)
    if f1 > meilleur_f1:
        meilleur_f1 = f1
        torch.save(modele_RN50_final.state_dict(), 'meilleur_modele_ResNet50_ad.pth')
        print("Nouveau meilleur modèle baseline sauvegardé !")
        patience_counter = 0
    else: 
        patience_counter+=1
    
    if patience_counter >= patience:
        print(f"\nEarly Stopping déclenché à l'époque {epoque+1}.")
        break

#### Test pour ResNet50

In [ ]:
modele_RN50_test_ad = models.resnet50(weights=None)

modele_RN50_test_ad.load_state_dict(torch.load('meilleur_modele_ResNet50_ad.pth', map_location=device))

nb_features_entree = modele_RN50_test_ad.fc.in_features
modele_RN50_test_ad.fc = nn.Sequential(
    nn.Dropout(p=meilleur_dropout), 
    nn.Linear(nb_features_entree, 2)
)
modele_RN50_test_ad = modele_RN50_test_ad.to(device)
modele_RN50_test_ad.eval()


predictions = []
vrais_labels = []

# torch.no_grad() coupe le calcul des gradients (économise la RAM et accélère le processus)
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        
        output = modele_RN50_test_ad(images)
        _, classes_predites = torch.max(output, 1)
        
        predictions.extend(classes_predites.cpu().numpy())
        vrais_labels.extend(labels.cpu().numpy())


acc_test = accuracy_score(vrais_labels, predictions)
rappel_test = recall_score(vrais_labels, predictions, pos_label=1, zero_division=0)
f1_test = f1_score(vrais_labels, predictions, pos_label=1, zero_division=0)


print(f"Accuracy Globale  : {acc_test*100:.2f}%")
print(f"Rappel (Sensibilité): {rappel_test*100:.2f}% ")
print(f"F1-Score          : {f1_test*100:.2f}%")


# Affichage du rapport détaillé de Scikit-Learn
noms_classes = ['Sain (0)', 'Fracture (1)']
print("\nRAPPORT DE CLASSIFICATION :")
print(classification_report(vrais_labels, predictions, target_names=noms_classes))


cm = confusion_matrix(vrais_labels, predictions)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=noms_classes, 
            yticklabels=noms_classes,
            annot_kws={"size": 14})

plt.title('Matrice de Confusion (Jeu de Test)', fontsize=16, pad=20)
plt.ylabel('Vérité', fontsize=12)
plt.xlabel('Prédiction', fontsize=12)
plt.show()

if device.type == "xpu":
    torch.xpu.empty_cache()
elif device.type == "cuda":
    torch.cuda.empty_cache()

#### Optuna pour DenseNet

In [ ]:
EPOQUES_PAR_TRIAL = 4
train_loader, val_loader, test_loader = preparer_dataloaders(img_size=(224, 224), batch_size=16) # bath size à ajuster mais plus petit à cause de densnet

def objective_densenet(trial):
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.7)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)
    freeze_layer = trial.suggest_categorical("freeze_layer", ["classifier_only", "denseblock4"])

    modele = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
    
    unfreeze_from = {
        "classifier_only": ["classifier."],
        "denseblock4": ["features.denseblock4.", "features.norm5.", "classifier."]
    }
    prefixes = unfreeze_from[freeze_layer]

    for name, param in modele.named_parameters():
        param.requires_grad = any(name.startswith(p) for p in prefixes)

    nb_features = modele.classifier.in_features
    modele.classifier = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(nb_features, 2))
    modele = modele.to(device)

    optimiseur = optim.Adam(filter(lambda p: p.requires_grad, modele.parameters()), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    for epoque in range(EPOQUES_PAR_TRIAL):
        modele.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimiseur.zero_grad()
            perte = criterion(modele(images), labels)
            perte.backward()
            optimiseur.step()
            
        modele.eval()
        toutes_preds, tous_labels = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                _, preds = torch.max(modele(images), 1)
                toutes_preds.extend(preds.cpu().numpy())
                tous_labels.extend(labels.cpu().numpy())
                
        f1 = f1_score(tous_labels, toutes_preds, pos_label=1, zero_division=0)
        trial.report(f1, epoque)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return f1


study_dense = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner(n_warmup_steps=1))
study_dense.optimize(objective_densenet, n_trials=15)
print(f"Meilleur F1 DenseNet-121 : {study_dense.best_value*100:.2f}% | Params: {study_dense.best_params}")

#### Entrainement de DenseNet avec les meilleurs hyperparamètres trouvés

In [ ]:
params_optimaux = study_dense.best_params

meilleur_lr = params_optimaux["lr"]
meilleur_dropout = params_optimaux["dropout_rate"]
meilleur_wd = params_optimaux["weight_decay"]
meilleur_degel = params_optimaux["freeze_layer"]

print(f"   -> Learning Rate : {meilleur_lr:.2e}")
print(f"   -> Dropout       : {meilleur_dropout:.2f}")
print(f"   -> Weight Decay  : {meilleur_wd:.2e}")
print(f"   -> Profondeur    : {meilleur_degel}")

# On utilise l'usine à DataLoaders avec un batch_size de 16 
train_loader, val_loader, test_loader = preparer_dataloaders(img_size=(224, 224), batch_size=16)

modele = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)

unfreeze_from = {
    "classifier_only": ["classifier."],
    "denseblock4": ["features.denseblock4.", "features.norm5.", "classifier."]
}
prefixes = unfreeze_from[meilleur_degel]

# Gel / Dégel dynamique
for name, param in modele.named_parameters():
    param.requires_grad = any(name.startswith(p) for p in prefixes)

# Nouvelle tête de classification (spécifique à DenseNet : .classifier)
nb_features = modele.classifier.in_features
modele.classifier = nn.Sequential(
    nn.Dropout(p=meilleur_dropout), 
    nn.Linear(nb_features, 2)
)
modele = modele.to(device)

# Optimiseur avec les paramètres Optuna
params_a_entrainer = filter(lambda p: p.requires_grad, modele.parameters())
optimiseur = optim.Adam(params_a_entrainer, lr=meilleur_lr, weight_decay=meilleur_wd)

# Loss standard (le WeightedRandomSampler gère déjà le déséquilibre)
criterion = nn.CrossEntropyLoss()

# Le cerveau dynamique : coupe le LR par 2 si le F1 stagne 2 époques de suite
scheduler = optim.ReduceLROnPlateau(optimiseur, mode='max', patience=2, factor=0.5, verbose=True)

# Entrainement
epoques_finales = 25
patience_early_stopping = 4 
patience_counter = 0
meilleur_f1 = 0.0


for epoque in range(epoques_finales):
    
    modele.train()
    perte_train_totale = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimiseur.zero_grad()
        output = modele(images)
        perte = criterion(output, labels)
        perte.backward()
        optimiseur.step()
        
        perte_train_totale += perte.item()
        
    moyenne_perte_train = perte_train_totale / len(train_loader)
    
    # Validation
    modele.eval()
    perte_val_totale = 0.0
    predictions_tot = []
    vrais_labels = []
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            output = modele(images)
            perte = criterion(output, labels)
            perte_val_totale += perte.item()
            
            _, classes_predites = torch.max(output, 1)
            predictions_tot.extend(classes_predites.cpu().numpy())
            vrais_labels.extend(labels.cpu().numpy())
            
    moyenne_perte_val = perte_val_totale / len(val_loader)
    
    acc = accuracy_score(vrais_labels, predictions_tot)
    rappel = recall_score(vrais_labels, predictions_tot, pos_label=1, zero_division=0)
    f1 = f1_score(vrais_labels, predictions_tot, pos_label=1, zero_division=0)
    
    lr_actuel = optimiseur.param_groups[0]['lr']
    print(f"Époque [{epoque+1:02d}/{epoques_finales}] | LR: {lr_actuel:.2e}")
    print(f"Train Loss: {moyenne_perte_train:.4f} | Val Loss: {moyenne_perte_val:.4f}")
    print(f"Acc: {acc*100:.1f}% | Rappel: {rappel*100:.1f}% | F1-Score: {f1*100:.2f}%")

    scheduler.step(f1) # On passe la note finale de l'époque au scheduler
    
    # Early stopping et sauvegarde
    if f1 > meilleur_f1:
        meilleur_f1 = f1
        torch.save(modele.state_dict(), 'densenet121_final.pth')
        print("Nouveau DenseNet sauvegardé.")
        patience_counter = 0 
    else:
        patience_counter += 1
        print(f" Pas d'amélioration du F1. Patience : {patience_counter}/{patience_early_stopping}")
        
    if patience_counter >= patience_early_stopping:
        print(f"\nEarly Stopping.")
        break
        
    print("-" * 70)

print(f"\n Terminé F1: {meilleur_f1*100:.2f}%)")

#### Test du meilleur modèle sur le jeu de test 

In [ ]:
modele_test = models.densenet121(weights=None) 

nb_features = modele_test.classifier.in_features
modele_test.classifier = nn.Sequential(
    nn.Dropout(p=0.5), # La valeur exacte du dropout Optuna importe peu en mode .eval(), 
                       # mais la structure Sequential doit être la même !
    nn.Linear(nb_features, 2)
)

modele_test.load_state_dict(torch.load('densenet121_final.pth', map_location=device))
modele_test = modele_test.to(device)
modele_test.eval()

predictions = []
vrais_labels = []

# torch.no_grad() coupe le calcul des gradients
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        
        output = modele_test(images)
        _, classes_predites = torch.max(output, 1)
        
        predictions.extend(classes_predites.cpu().numpy())
        vrais_labels.extend(labels.cpu().numpy())


acc_test = accuracy_score(vrais_labels, predictions)
rappel_test = recall_score(vrais_labels, predictions, pos_label=1, zero_division=0)
f1_test = f1_score(vrais_labels, predictions, pos_label=1, zero_division=0)


print(f"Accuracy Globale  : {acc_test*100:.2f}%")
print(f"Rappel (Sensibilité): {rappel_test*100:.2f}% ")
print(f"F1-Score          : {f1_test*100:.2f}%")


# Affichage du rapport détaillé de Scikit-Learn
noms_classes = ['Sain (0)', 'Fracture (1)']
print("\nRAPPORT DE CLASSIFICATION :")
print(classification_report(vrais_labels, predictions, target_names=noms_classes))


cm = confusion_matrix(vrais_labels, predictions)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=noms_classes, 
            yticklabels=noms_classes,
            annot_kws={"size": 14})

plt.title('Matrice de Confusion (Jeu de Test)', fontsize=16, pad=20)
plt.ylabel('Vérité', fontsize=12)
plt.xlabel('Prédiction', fontsize=12)
plt.show()

if device.type == "xpu":
    torch.xpu.empty_cache()
elif device.type == "cuda":
    torch.cuda.empty_cache()

#### Optuna pour EfficientNet V2S

In [ ]:
EPOQUES_PAR_TRIAL = 4
train_loader, val_loader, test_loader = preparer_dataloaders(img_size=(384, 384), batch_size=16) # taille d'images plus grandes

def objective_efficientnet(trial):
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.7)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)
    freeze_layer = trial.suggest_categorical("freeze_layer", ["classifier_only", "stage7", "stage6_7"])

    modele = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
    
    unfreeze_from = {
        "classifier_only": ["classifier."],
        "stage7": ["features.7.", "classifier."],
        "stage6_7": ["features.6.", "features.7.", "classifier."]
    }
    prefixes = unfreeze_from[freeze_layer]

    for name, param in modele.named_parameters():
        param.requires_grad = any(name.startswith(p) for p in prefixes)

    # EfficientNet a déjà un Dropout par défaut dans classifier[0], on cible le Linear dans classifier[1]
    nb_features = modele.classifier[1].in_features
    modele.classifier = nn.Sequential(nn.Dropout(p=dropout_rate), nn.Linear(nb_features, 2))
    modele = modele.to(device)

    optimiseur = optim.Adam(filter(lambda p: p.requires_grad, modele.parameters()), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    for epoque in range(EPOQUES_PAR_TRIAL):
        modele.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimiseur.zero_grad()
            perte = criterion(modele(images), labels)
            perte.backward()
            optimiseur.step()
            
        modele.eval()
        predictions_tot, vrais_labels = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                _, preds = torch.max(modele(images), 1)
                predictions_tot.extend(preds.cpu().numpy())
                vrais_labels.extend(labels.cpu().numpy())
                
        f1 = f1_score(vrais_labels, predictions_tot, pos_label=1, zero_division=0)
        trial.report(f1, epoque)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return f1

study_effnet = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner(n_warmup_steps=1))
study_effnet.optimize(objective_efficientnet, n_trials=15)
print(f" Meilleur F1 EfficientNet-V2-S : {study_effnet.best_value*100:.2f}% | Params: {study_effnet.best_params}")

#### Entrainement de EfficientNet V2S avec les meilleurs hyperparamètres trouvés

In [ ]:
params_optimaux = study_effnet.best_params

meilleur_lr = params_optimaux["lr"]
meilleur_dropout = params_optimaux["dropout_rate"]
meilleur_wd = params_optimaux["weight_decay"]
meilleur_degel = params_optimaux["freeze_layer"]

print(f"   -> Learning Rate : {meilleur_lr:.2e}")
print(f"   -> Dropout       : {meilleur_dropout:.2f}")
print(f"   -> Weight Decay  : {meilleur_wd:.2e}")
print(f"   -> Profondeur    : {meilleur_degel}")

# Ici on change la taille d'entrée des images pour avoir une résolution plus élevé en 384*384
train_loader, val_loader, test_loader = preparer_dataloaders(img_size=(384, 384), batch_size=16)



modele = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)

unfreeze_from = {
    "classifier_only": ["classifier."],
    "stage7": ["features.7.", "classifier."],
    "stage6_7": ["features.6.", "features.7.", "classifier."]
}
prefixes = unfreeze_from[meilleur_degel]

# Gel / Dégel dynamique
for name, param in modele.named_parameters():
    param.requires_grad = any(name.startswith(p) for p in prefixes)

# Nouvelle tête de classification
# Dans EfficientNet, le Linear d'origine est en position [1] du bloc classifier
nb_features = modele.classifier[1].in_features
modele.classifier = nn.Sequential(
    nn.Dropout(p=meilleur_dropout), 
    nn.Linear(nb_features, 2)
)
modele = modele.to(device)


params_a_entrainer = filter(lambda p: p.requires_grad, modele.parameters())
optimiseur = optim.Adam(params_a_entrainer, lr=meilleur_lr, weight_decay=meilleur_wd)

# Toujours CrossEntropy standard car le Sampler gère le déséquilibre en amont
criterion = nn.CrossEntropyLoss()

# Le Scheduler pour affiner l'approche du minimum global
scheduler = optim.ReduceLROnPlateau(optimiseur, mode='max', patience=2, factor=0.5, verbose=True)


epoques_finales = 25
patience_early_stopping = 5 
patience_counter = 0
meilleur_f1 = 0.0

# Entrainement

for epoque in range(epoques_finales):
    
    modele.train()
    perte_train_totale = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimiseur.zero_grad()
        output = modele(images)
        perte = criterion(output, labels)
        perte.backward()
        optimiseur.step()
        
        perte_train_totale += perte.item()
        
    moyenne_perte_train = perte_train_totale / len(train_loader)
    
    # Validation
    modele.eval()
    perte_val_totale = 0.0
    predictions_tot = []
    vrais_labels = []
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            output = modele(images)
            perte = criterion(output, labels)
            perte_val_totale += perte.item()
            
            _, classes_predites = torch.max(output, 1)
            predictions_tot.extend(classes_predites.cpu().numpy())
            vrais_labels.extend(labels.cpu().numpy())
            
    moyenne_perte_val = perte_val_totale / len(val_loader)
    
    acc = accuracy_score(vrais_labels, predictions_tot)
    rappel = recall_score(vrais_labels, predictions_tot, pos_label=1, zero_division=0)
    f1 = f1_score(vrais_labels, predictions_tot, pos_label=1, zero_division=0)

    lr_actuel = optimiseur.param_groups[0]['lr']
    print(f"Époque [{epoque+1:02d}/{epoques_finales}] | LR: {lr_actuel:.2e}")
    print(f"Train Loss: {moyenne_perte_train:.4f} | Val Loss: {moyenne_perte_val:.4f}")
    print(f"Acc: {acc*100:.1f}% | Rappel: {rappel*100:.1f}% | F1-Score: {f1*100:.2f}%")
    

    scheduler.step(f1) # on donne le résultat du F1-score au scheduler 
    
    # early stopping et sauvegarde
    if f1 > meilleur_f1:
        meilleur_f1 = f1
        torch.save(modele.state_dict(), 'efficientnet_v2_s_final.pth')
        print("EfficientNet sauvegardé.")
        patience_counter = 0 
    else:
        patience_counter += 1
        print(f"Pas d'amélioration du F1. Patience : {patience_counter}/{patience_early_stopping}")
        
    if patience_counter >= patience_early_stopping:
        print(f"\nEarly Stopping")
        break
        
    print("-" * 70)

print(f"\nEfficientNet-V2-S meilleur F1-score = {meilleur_f1*100:.2f}%)")

#### Test de EfficientNet V2s sur les données de test

In [ ]:
modele_V2S_test = models.efficientnet_v2_s(weights=None) 

nb_features = modele_V2S_test.classifier[1].in_features
modele_V2S_test.classifier = nn.Sequential(
    nn.Dropout(p=meilleur_dropout), 
    nn.Linear(nb_features, 2)
)


modele_V2S_test.load_state_dict(torch.load('efficientnet_v2_s_final.pth', map_location=device))
modele_V2S_test = modele_V2S_test.to(device)
modele_V2S_test.eval()

predictions = []
vrais_labels = []

# torch.no_grad() coupe le calcul des gradients
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        
        output = modele_V2S_test(images)
        _, classes_predites = torch.max(output, 1)
        
        predictions.extend(classes_predites.cpu().numpy())
        vrais_labels.extend(labels.cpu().numpy())


acc_test = accuracy_score(vrais_labels, predictions)
rappel_test = recall_score(vrais_labels, predictions, pos_label=1, zero_division=0)
f1_test = f1_score(vrais_labels, predictions, pos_label=1, zero_division=0)


print(f"Accuracy Globale  : {acc_test*100:.2f}%")
print(f"Rappel (Sensibilité): {rappel_test*100:.2f}% ")
print(f"F1-Score          : {f1_test*100:.2f}%")


# Affichage du rapport détaillé de Scikit-Learn
noms_classes = ['Sain (0)', 'Fracture (1)']
print("\nRAPPORT DE CLASSIFICATION :")
print(classification_report(vrais_labels, predictions, target_names=noms_classes))


cm = confusion_matrix(vrais_labels, predictions)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=noms_classes, 
            yticklabels=noms_classes,
            annot_kws={"size": 14})

plt.title('Matrice de Confusion (Jeu de Test)', fontsize=16, pad=20)
plt.ylabel('Vérité', fontsize=12)
plt.xlabel('Prédiction', fontsize=12)
plt.show()

if device.type == "xpu":
    torch.xpu.empty_cache()
elif device.type == "cuda":
    torch.cuda.empty_cache()

Sachant que ce dataset contient un fichier YOLO (You Look Only Once), il serait vraiment intéressant de tester un modèle de yolo en inférence. Nous avons choisi YOLOv8n

In [ ]:
from ultralytics import YOLO

# Load a pretrained YOLO26n model
model = YOLO("yolo26n.pt")

# Train the model on the COCO8 dataset for 100 epochs
train_results = model.train(
    data="coco8.yaml",  # Path to dataset configuration file
    epochs=100,  # Number of training epochs
    imgsz=640,  # Image size for training
    device="cpu",  # Device to run on (e.g., 'cpu', 0, [0,1,2,3])
)

# Evaluate the model's performance on the validation set
metrics = model.val()

# Perform object detection on an image
results = model("path/to/image.jpg")  # Predict on an image
results[0].show()  # Display results

# Export the model to ONNX format for deployment
#path = model.export(format="onnx")  # Returns the path to the exported model